# Session 3: Databases, SQL, and Django ORM

This notebook covers relational database concepts, basic SQL commands, mapping SQL to Django ORM, and a practical exercise to create models with a One-to-Many relationship, insert data, and query using SQL and ORM.

## Section 1: Basic Database Concepts

### Relational Database
- A **relational database** organizes data into tables with rows and columns, using relationships to link data.
- Data is stored in a structured format, ensuring consistency and enabling efficient querying.

### Tables, Rows, Columns, Keys
- **Table**: A collection of data organized into rows and columns (e.g., a table for `Products`).
- **Row**: A single record in a table (e.g., one product).
- **Column**: A field that stores a specific attribute for all rows (e.g., `name`, `price`).
- **Primary Key**: A unique identifier for each row in a table (e.g., `product_id`).
- **Foreign Key**: A column in one table that references the primary key of another table, establishing a relationship.

### Normalization (up to 2NF)
- **Normalization**: The process of organizing data to reduce redundancy and improve integrity.
- **1NF (First Normal Form)**: Ensure each column contains atomic values, and each row is unique (via a primary key).
- **2NF (Second Normal Form)**: Meet 1NF and ensure non-key columns depend on the entire primary key, not just part of it (applies to tables with composite keys).

### Types of Relationships
- **One-to-One**: One record in a table corresponds to exactly one record in another (e.g., a user and their profile).
- **One-to-Many**: One record in a table can relate to multiple records in another (e.g., one category has many products).
- **Many-to-Many**: Multiple records in one table relate to multiple records in another (e.g., students and courses, implemented via a junction table).

## Section 2: Basic SQL Commands

We'll demonstrate SQL commands using a `Category` and `Product` table as examples.

### CREATE TABLE
Creates a new table with defined columns and constraints.

In [ ]:
# SQL: Create Category and Product tables
CREATE TABLE Category (
    id INTEGER PRIMARY KEY,
    name VARCHAR(50) NOT NULL
);

CREATE TABLE Product (
    id INTEGER PRIMARY KEY,
    name VARCHAR(100) NOT NULL,
    price DECIMAL(10, 2),
    category_id INTEGER,
    FOREIGN KEY (category_id) REFERENCES Category(id)
);

### INSERT INTO
Inserts data into a table.

In [ ]:
# SQL: Insert data
INSERT INTO Category (id, name) VALUES (1, 'Electronics');
INSERT INTO Product (id, name, price, category_id) VALUES (1, 'Laptop', 999.99, 1);

### SELECT
Queries data from a table.

In [ ]:
# SQL: Select all products
SELECT * FROM Product;

# SQL: Filter with WHERE
SELECT * FROM Product WHERE price > 500;

# SQL: Sort with ORDER BY
SELECT * FROM Product ORDER BY price DESC;

### UPDATE and DELETE
Modifies or removes data.

In [ ]:
# SQL: Update a product's price
UPDATE Product SET price = 1099.99 WHERE id = 1;

# SQL: Delete a product
DELETE FROM Product WHERE id = 1;

### Aggregate Functions
Performs calculations on data.

In [ ]:
# SQL: Count products, average price, etc.
SELECT COUNT(*) AS total_products, AVG(price) AS avg_price, MAX(price) AS max_price
FROM Product;

### INNER JOIN
Combines rows from two tables based on a related column.

In [ ]:
# SQL: Join Product and Category
SELECT Product.name, Product.price, Category.name AS category_name
FROM Product
INNER JOIN Category ON Product.category_id = Category.id;

## Section 3: Mapping SQL to Django ORM

Django’s ORM (Object-Relational Mapping) allows writing Python code instead of SQL to interact with the database.

### Model.objects.all() ⇔ SELECT * FROM
Retrieves all records from a model.

In [ ]:
# ORM: Equivalent to SELECT * FROM Product
from dashboard.models import Product
Product.objects.all()

### Model.objects.filter(...) ⇔ WHERE
Filters records based on conditions.

In [ ]:
# ORM: Equivalent to SELECT * FROM Product WHERE price > 500
Product.objects.filter(price__gt=500)

### Model.objects.get(...)
Retrieves a single record matching the criteria.

In [ ]:
# ORM: Equivalent to SELECT * FROM Product WHERE id = 1
Product.objects.get(id=1)

### annotate and aggregate
Performs calculations on querysets.

In [ ]:
# ORM: Equivalent to SELECT COUNT(*), AVG(price)
from django.db.models import Count, Avg, Max
Product.objects.aggregate(total=Count('id'), avg_price=Avg('price'), max_price=Max('price'))

# ORM: Annotate with product count per category
from django.db.models import Count
Category.objects.annotate(product_count=Count('product'))

## Section 4: Practical Exercise

### Task: Create Two Models with One-to-Many Relationship
Create `Category` and `Product` models in the `dashboard` app with a One-to-Many relationship.

**Action**: Update `dashboard/models.py` to define the models.

In [ ]:
%%writefile dashboard/models.py
from django.db import models

class Category(models.Model):
    name = models.CharField(max_length=50)

    def __str__(self):
        return self.name

class Product(models.Model):
    name = models.CharField(max_length=100)
    price = models.DecimalField(max_digits=10, decimal_places=2)
    category = models.ForeignKey(Category, on_delete=models.CASCADE, related_name='products')

    def __str__(self):
        return self.name

### Apply Migrations
Generate and apply migrations to create the database tables.

**Action**: Run `makemigrations` and `migrate`.

In [ ]:
!python manage.py makemigrations
!python manage.py migrate

### Insert Data via Django Shell
Insert data into `Category` and `Product` models using the Django shell.

**Action**: Run these commands in the Django shell (`python manage.py shell`). Below is the equivalent code.

In [ ]:
# Run in Django shell: python manage.py shell
from dashboard.models import Category, Product

# Create categories
cat1 = Category.objects.create(name='Electronics')
cat2 = Category.objects.create(name='Books')

# Create products
Product.objects.create(name='Laptop', price=999.99, category=cat1)
Product.objects.create(name='Laptop', price=999.99, category=cat1)
Product.objects.create(name='Python Book', price=29.99, category=cat2)

### Write Equivalent SQL and ORM Queries
Query the data using both SQL and Django ORM.

**Query 1**: Get all products with their category names.

In [ ]:
# SQL
SELECT Product.name, Product.price, Category.name AS category_name
FROM Product
INNER JOIN Category ON Product.category_id = Category.id;

# ORM
from dashboard.models import Product
Product.objects.select_related('category').values('name', 'price', 'category__name')

**Query 2**: Get products with price greater than 500.

In [ ]:
# SQL
SELECT * FROM Product WHERE price > 500;

# ORM
Product.objects.filter(price__gt=500)

**Query 3**: Count products per category.

In [ ]:
# SQL
SELECT Category.name, COUNT(Product.id) AS product_count
FROM Category
LEFT JOIN Product ON Category.id = Product.category_id
GROUP BY Category.id;

# ORM
from django.db.models import Count
Category.objects.annotate(product_count=Count('products')).values('name', 'product_count')

## Exercise Summary

You have:
1. Created `Category` and `Product` models with a One-to-Many relationship.
2. Applied migrations to create the database tables.
3. Inserted sample data via the Django shell.
4. Written equivalent SQL and ORM queries to retrieve data.

**Test the Setup**:
- Run `python manage.py shell` and execute the data insertion commands.
- Use the ORM queries in the shell to verify the results.
- Optionally, use a database tool (e.g., SQLite’s `sqlite3` or a GUI like DB Browser) to run the SQL queries.